# 7.2 Self-Attention from Scratch

這份 Notebook 用 TensorFlow 手刻 scaled dot-product self-attention，並和 Keras `MultiHeadAttention` 對照。


## 1. 載入套件


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(42)


## 2. 建立一批 token embeddings


In [ ]:
batch_size = 2
seq_len = 5
embed_dim = 8
x = tf.random.normal((batch_size, seq_len, embed_dim), seed=42)
print('input shape:', x.shape)


## 3. Q、K、V projection


In [ ]:
q_proj = tf.keras.layers.Dense(embed_dim, use_bias=False)
k_proj = tf.keras.layers.Dense(embed_dim, use_bias=False)
v_proj = tf.keras.layers.Dense(embed_dim, use_bias=False)

Q = q_proj(x)
K = k_proj(x)
V = v_proj(x)
print(Q.shape, K.shape, V.shape)


## 4. 手刻 scaled dot-product attention


In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = tf.cast(tf.shape(K)[-1], tf.float32)
    scores = tf.matmul(Q, K, transpose_b=True) / tf.sqrt(d_k)
    if mask is not None:
        scores += (mask * -1e9)
    weights = tf.nn.softmax(scores, axis=-1)
    output = tf.matmul(weights, V)
    return output, weights

output, weights = scaled_dot_product_attention(Q, K, V)
print('output:', output.shape)
print('weights:', weights.shape)


## 5. 視覺化第一筆資料的 attention weights


In [ ]:
plt.figure(figsize=(4.5, 4))
plt.imshow(weights[0].numpy(), cmap='Blues')
plt.xlabel('Key position')
plt.ylabel('Query position')
plt.colorbar(label='weight')
plt.title('Self-attention weights')
plt.show()


## 6. 加入 causal mask 範例

Causal mask 會阻止目前位置看到未來位置。


In [ ]:
causal_mask = 1 - tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
masked_output, masked_weights = scaled_dot_product_attention(Q, K, V, causal_mask)
print('causal mask:\n', causal_mask.numpy().astype(int))
print('masked weights for first sample:\n', masked_weights[0].numpy().round(3))


## 7. 對照 Keras MultiHeadAttention


In [ ]:
mha = tf.keras.layers.MultiHeadAttention(num_heads=2, key_dim=4)
mha_output = mha(x, x)
print('MultiHeadAttention output shape:', mha_output.shape)


## 8. 小結

Self-attention 的輸入輸出 shape 都是 `(batch, sequence_length, embedding_dim)`。手刻版能看懂 Q、K、V 與權重；正式模型通常直接使用 `MultiHeadAttention`。
